In [11]:
# Cell 1: Build CH4 fermionic Hamiltonian and Hermitian fermionic terms

import pandas as pd
import numpy as np

from openfermion.chem import MolecularData
from openfermion.ops import FermionOperator
from openfermion.transforms import get_fermion_operator, normal_ordered
from openfermion.utils import hermitian_conjugated
from openfermionpyscf import run_pyscf

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Methane / CH4 configuration
# ------------------------------------------------------------

CH_BOND_LENGTH = 1.087  # Angstrom, approximate equilibrium C-H bond length
BASIS = "sto-3g"
MULTIPLICITY = 1
CHARGE = 0

# Full CH4/STO-3G should give 18 qubits:
# C has 5 STO-3G spatial orbitals, each H has 1, so 5 + 4 = 9 spatial orbitals.
# 9 spatial orbitals * 2 spin orbitals = 18 qubits.
USE_ACTIVE_SPACE = True

# Optional active-space example:
# Freeze carbon 1s-like core spatial orbital 0 and keep valence active orbitals.
# This would usually reduce CH4/STO-3G from 18 qubits to 16 qubits.
OCCUPIED_INDICES = [0]
ACTIVE_INDICES = [1, 2, 3, 4, 5, 6, 7, 8]

# Set this smaller, e.g. 1e-10, if tiny numerical terms clutter the graph.
TERM_ABS_TOL = 1e-12


def format_fermion_term(term):
    if term == ():
        return "I"

    pieces = []
    for orbital, action in term:
        if action == 1:
            pieces.append(f"a_{orbital}^dagger")
        else:
            pieces.append(f"a_{orbital}")
    return " ".join(pieces)


def sort_fermion_key(term):
    return (len(term), term)


def coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_fermion_key(item[0])):
        pieces.append(f"{coeff_to_str(coeff, digits)} {format_fermion_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def dagger_term_key(term):
    """
    Return the OpenFermion key for O^dagger, where O is one monomial.
    """
    O = FermionOperator(term, 1.0)
    O_dag = normal_ordered(hermitian_conjugated(O))
    O_dag.compress(abs_tol=TERM_ABS_TOL)

    if len(O_dag.terms) != 1:
        raise ValueError(f"Expected one dagger term, got: {O_dag}")

    return next(iter(O_dag.terms.keys()))


def make_hermitian_fermionic_terms(fermion_hamiltonian, tol=TERM_ABS_TOL):
    """
    Group raw monomials into Hermitian fermionic Hamiltonian terms.

    If O is self-adjoint, keep c O.
    If O is not self-adjoint, group c O + c* O^dagger using the
    coefficients already present in the Hamiltonian.
    """
    used = set()
    hermitian_terms = []

    for term, coeff in fermion_hamiltonian.terms.items():
        if term in used:
            continue

        dag = dagger_term_key(term)

        if dag == term:
            T = FermionOperator(term, coeff)
            used.add(term)
        else:
            dag_coeff = fermion_hamiltonian.terms.get(dag, 0.0)

            T = FermionOperator(term, coeff)
            T += FermionOperator(dag, dag_coeff)

            used.add(term)
            used.add(dag)

        T = normal_ordered(T)
        T.compress(abs_tol=tol)
        hermitian_terms.append(T)

    return hermitian_terms


def infer_n_qubits_from_fermion_operator(op):
    """
    Infer the number of spin orbitals used by the FermionOperator.

    This is important for active-space Hamiltonians, where the active
    modes may be reindexed and smaller than molecule.n_qubits.
    """
    max_orbital = -1

    for term in op.terms:
        for orbital, action in term:
            max_orbital = max(max_orbital, orbital)

    return max_orbital + 1


def build_ch4_geometry(ch_bond_length=CH_BOND_LENGTH):
    """
    Build tetrahedral methane geometry.

    Carbon is placed at the origin.
    The four hydrogens are placed at tetrahedral directions:
        ( 1,  1,  1)
        ( 1, -1, -1)
        (-1,  1, -1)
        (-1, -1,  1)

    Each direction is normalized so the C-H distance equals ch_bond_length.
    """
    scale = ch_bond_length / np.sqrt(3.0)

    geometry = [
        ("C", (0.0, 0.0, 0.0)),
        ("H", ( scale,  scale,  scale)),
        ("H", ( scale, -scale, -scale)),
        ("H", (-scale,  scale, -scale)),
        ("H", (-scale, -scale,  scale)),
    ]

    return geometry


def build_ch4_fermionic_hamiltonian(
    ch_bond_length=CH_BOND_LENGTH,
    basis=BASIS,
    multiplicity=MULTIPLICITY,
    charge=CHARGE,
    use_active_space=USE_ACTIVE_SPACE,
    occupied_indices=OCCUPIED_INDICES,
    active_indices=ACTIVE_INDICES,
):
    """
    Build a CH4 fermionic Hamiltonian using OpenFermion + PySCF.

    If use_active_space=True, molecule.get_molecular_hamiltonian is called
    with occupied_indices and active_indices.
    """
    geometry = build_ch4_geometry(ch_bond_length=ch_bond_length)

    molecule = MolecularData(
        geometry=geometry,
        basis=basis,
        multiplicity=multiplicity,
        charge=charge,
        description=f"CH4_{ch_bond_length}",
    )

    # FCI is not required for constructing the fermionic Hamiltonian.
    # For CH4, keep run_fci=False.
    molecule = run_pyscf(
        molecule,
        run_scf=True,
        run_fci=False,
    )

    if use_active_space:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian(
            occupied_indices=occupied_indices,
            active_indices=active_indices,
        )
    else:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian()

    fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)
    fermion_hamiltonian = normal_ordered(fermion_hamiltonian)
    fermion_hamiltonian.compress(abs_tol=TERM_ABS_TOL)

    n_qubits = infer_n_qubits_from_fermion_operator(fermion_hamiltonian)

    return molecule, fermion_hamiltonian, n_qubits


# ------------------------------------------------------------
# Build CH4 fermionic Hamiltonian
# ------------------------------------------------------------

molecule, Hf, n_qubits = build_ch4_fermionic_hamiltonian()
hermitian_terms = make_hermitian_fermionic_terms(Hf)

print("Molecule: CH4 / Methane")
print("Basis:", BASIS)
print("C-H bond length [Angstrom]:", CH_BOND_LENGTH)
print("Use active space:", USE_ACTIVE_SPACE)
if USE_ACTIVE_SPACE:
    print("Frozen occupied spatial orbitals:", OCCUPIED_INDICES)
    print("Active spatial orbitals:", ACTIVE_INDICES)
print("Full molecule electrons:", molecule.n_electrons)
print("Full molecule spatial orbitals:", molecule.n_orbitals)
print("Full molecule spin orbitals / qubits:", molecule.n_qubits)
print("Hamiltonian spin orbitals / qubits used:", n_qubits)
print("Number of raw OpenFermion monomial terms:", len(Hf.terms))
print("Number of Hermitian fermionic terms:", len(hermitian_terms))

print("\n=== Full fermionic Hamiltonian H_f ===")
print(Hf)


# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

raw_rows = []

for idx, (term, coeff) in enumerate(
    sorted(Hf.terms.items(), key=lambda item: sort_fermion_key(item[0]))
):
    raw_rows.append(
        {
            "raw_index": idx,
            "coefficient": coeff_to_str(coeff),
            "monomial": format_fermion_term(term),
            "OpenFermion_key": term,
        }
    )

raw_df = pd.DataFrame(raw_rows)

print("\n=== Raw fermionic monomials c_alpha O_alpha ===")
display(raw_df)


hermitian_rows = []

for i, T in enumerate(hermitian_terms):
    hermitian_rows.append(
        {
            "vertex": f"T_{i}",
            "number_of_monomials": len(T.terms),
            "fermionic_term": operator_to_string(T),
        }
    )

hermitian_df = pd.DataFrame(hermitian_rows)

print("\n=== Hermitian fermionic terms T_i ===")
display(hermitian_df)

Molecule: CH4 / Methane
Basis: sto-3g
C-H bond length [Angstrom]: 1.087
Use active space: True
Frozen occupied spatial orbitals: [0]
Active spatial orbitals: [1, 2, 3, 4, 5, 6, 7, 8]
Full molecule electrons: 10
Full molecule spatial orbitals: 9
Full molecule spin orbitals / qubits: 18
Hamiltonian spin orbitals / qubits used: 16
Number of raw OpenFermion monomial terms: 4689
Number of Hermitian fermionic terms: 2413

=== Full fermionic Hamiltonian H_f ===
-22.408915115183795 [] +
-4.005144750335747 [0^ 0] +
-0.5329579619356528 [0^ 14] +
-0.5040433677317613 [1^ 0^ 1 0] +
-0.10834818702074449 [1^ 0^ 3 2] +
-0.10834818702074467 [1^ 0^ 5 4] +
-0.10834818702074432 [1^ 0^ 7 6] +
0.0038543796497326277 [1^ 0^ 8 3] +
-0.010819833086620264 [1^ 0^ 8 5] +
-0.014460132543415518 [1^ 0^ 8 7] +
-0.0038543796497326277 [1^ 0^ 9 2] +
0.010819833086620264 [1^ 0^ 9 4] +
0.014460132543415518 [1^ 0^ 9 6] +
-0.04762387544706916 [1^ 0^ 9 8] +
0.001996315980880118 [1^ 0^ 10 3] +
-0.014439923307331728 [1^ 0^ 10 5

,raw_index,coefficient,monomial,OpenFermion_key
0,0,-22.40891512,I,()
1,1,-4.00514475,a_0^dagger a_0,"((0, 1), (0, 0))"
2,2,-0.53295796,a_0^dagger a_14,"((0, 1), (14, 0))"
3,3,-4.00514475,a_1^dagger a_1,"((1, 1), (1, 0))"
4,4,-0.53295796,a_1^dagger a_15,"((1, 1), (15, 0))"
...,...,...,...,...
4684,4684,+0.00185219,a_15^dagger a_14^dagger a_13 a_6,"((15, 1), (14, 1), (13, 0), (6, 0))"
4685,4685,-0.10185535,a_15^dagger a_14^dagger a_13 a_12,"((15, 1), (14, 1), (13, 0), (12, 0))"
4686,4686,+0.06588115,a_15^dagger a_14^dagger a_14 a_1,"((15, 1), (14, 1), (14, 0), (1, 0))"
4687,4687,-0.06588115,a_15^dagger a_14^dagger a_15 a_0,"((15, 1), (14, 1), (15, 0), (0, 0))"



=== Hermitian fermionic terms T_i ===


,vertex,number_of_monomials,fermionic_term
0,T_0,1,-22.40891512 I
1,T_1,1,-4.00514475 a_0^dagger a_0
2,T_2,2,-0.53295796 a_0^dagger a_14 + -0.53295796 a_14^dagger a_0
3,T_3,1,-4.00514475 a_1^dagger a_1
4,T_4,2,-0.53295796 a_1^dagger a_15 + -0.53295796 a_15^dagger a_1
...,...,...,...
2408,T_2408,2,+0.10185535 a_14^dagger a_13^dagger a_15 a_12 + +0.10185535 a_15^dagger a_12^dagger a_14 a_13
2409,T_2409,1,-0.47820478 a_15^dagger a_12^dagger a_15 a_12
2410,T_2410,1,-0.47820478 a_14^dagger a_13^dagger a_14 a_13
2411,T_2411,1,-0.37634943 a_15^dagger a_13^dagger a_15 a_13


In [12]:
# Cell 2: Build the H2 fermionic noncommutation graph

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


def fermionic_commutator(A, B, tol=1e-12):
    """
    Compute [A, B] = AB - BA directly in the fermionic algebra.
    """
    C = normal_ordered(A * B - B * A)
    C.compress(abs_tol=tol)
    return C


def commute(A, B, tol=1e-12):
    """
    Return True if [A, B] = 0.
    """
    C = fermionic_commutator(A, B, tol=tol)
    return len(C.terms) == 0


# ------------------------------------------------------------
# Build noncommutation graph
# ------------------------------------------------------------
# Vertex i = Hermitian fermionic term T_i
# Edge (i, j) exists if [T_i, T_j] != 0

G = nx.Graph()

for i, T in enumerate(hermitian_terms):
    G.add_node(
        i,
        label=f"T_{i}",
        operator=T,
        operator_string=operator_to_string(T),
        number_of_monomials=len(T.terms),
    )

for i in range(len(hermitian_terms)):
    for j in range(i + 1, len(hermitian_terms)):
        Cij = fermionic_commutator(hermitian_terms[i], hermitian_terms[j])

        if len(Cij.terms) != 0:
            G.add_edge(
                i,
                j,
                commutator=Cij,
                commutator_string=operator_to_string(Cij),
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_vertices = G.number_of_nodes()
n_total_pairs = n_vertices * (n_vertices - 1) // 2
n_noncommuting_pairs = G.number_of_edges()
n_commuting_pairs = n_total_pairs - n_noncommuting_pairs

print("=== Fermionic noncommutation graph summary ===")
print("Number of vertices / fermionic terms:", n_vertices)
print("Number of total unordered pairs:", n_total_pairs)
print("Number of noncommuting pairs / edges:", n_noncommuting_pairs)
print("Number of commuting pairs:", n_commuting_pairs)
print("Is graph bipartite?", nx.is_bipartite(G))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G.degree[i],
            "commutes_with_all": G.degree[i] == 0,
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df = pd.DataFrame(vertex_rows)

print("\n=== Vertices: fermionic terms ===")
display(vertex_df)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": f"[T_{i}, T_{j}] != 0",
            "commutator": data["commutator_string"],
        }
    )

edge_df = pd.DataFrame(edge_rows)

print("\n=== Edges: noncommuting pairs ===")
display(edge_df)


# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G)

# node_labels = {
#     i: f"T_{i}"
#     for i in G.nodes()
# }

# node_sizes = [
#     1000 + 250 * G.degree[i]
#     for i in G.nodes()
# ]

# nx.draw_networkx_nodes(G, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G, pos, width=1.5)
# nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=11, font_weight="bold")

# plt.title("H2 Fermionic Noncommutation Graph")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph summary ===
Number of vertices / fermionic terms: 2413
Number of total unordered pairs: 2910078
Number of noncommuting pairs / edges: 1663978
Number of commuting pairs: 1246100
Is graph bipartite? False

=== Vertices: fermionic terms ===


,vertex,degree,commutes_with_all,fermionic_term
0,T_0,0,True,-22.40891512 I
1,T_1,428,False,-4.00514475 a_0^dagger a_0
2,T_2,838,False,-0.53295796 a_0^dagger a_14 + -0.53295796 a_14^dagger a_0
3,T_3,428,False,-4.00514475 a_1^dagger a_1
4,T_4,838,False,-0.53295796 a_1^dagger a_15 + -0.53295796 a_15^dagger a_1
...,...,...,...,...
2408,T_2408,1424,False,+0.10185535 a_14^dagger a_13^dagger a_15 a_12 + +0.10185535 a_15^dagger a_12^dagger a_14 a_13
2409,T_2409,817,False,-0.47820478 a_15^dagger a_12^dagger a_15 a_12
2410,T_2410,817,False,-0.47820478 a_14^dagger a_13^dagger a_14 a_13
2411,T_2411,785,False,-0.37634943 a_15^dagger a_13^dagger a_15 a_13



=== Edges: noncommuting pairs ===


,source,target,meaning,commutator
0,T_1,T_2,"[T_1, T_2] != 0",+2.13457378 a_0^dagger a_14 + -2.13457378 a_14^dagger a_0
1,T_1,T_38,"[T_1, T_38] != 0",-0.35109953 a_1^dagger a_0^dagger a_14 a_1 + +0.35109953 a_14^dagger a_1^dagger a_1 a_0
2,T_1,T_39,"[T_1, T_39] != 0",+0.43395017 a_1^dagger a_0^dagger a_3 a_2 + -0.43395017 a_3^dagger a_2^dagger a_1 a_0
3,T_1,T_40,"[T_1, T_40] != 0",-0.01543735 a_1^dagger a_0^dagger a_8 a_3 + +0.01543735 a_8^dagger a_3^dagger a_1 a_0
4,T_1,T_41,"[T_1, T_41] != 0",-0.00799553 a_1^dagger a_0^dagger a_10 a_3 + +0.00799553 a_10^dagger a_3^dagger a_1 a_0
...,...,...,...,...
1663973,T_2405,T_2410,"[T_2405, T_2410] != 0",+0.00914893 a_14^dagger a_13^dagger a_12^dagger a_15 a_14 a_12 + -0.00914893 a_15^dagger a_14^dagger a_12^dagger a_14 a_13 a_12
1663974,T_2405,T_2412,"[T_2405, T_2412] != 0",-0.00884702 a_14^dagger a_13^dagger a_12^dagger a_15 a_14 a_12 + +0.00884702 a_15^dagger a_14^dagger a_12^dagger a_14 a_13 a_12
1663975,T_2406,T_2412,"[T_2406, T_2412] != 0",-0.04710039 a_13^dagger a_12^dagger a_15 a_14 + +0.04710039 a_15^dagger a_14^dagger a_13 a_12
1663976,T_2408,T_2409,"[T_2408, T_2409] != 0",+0.04870772 a_14^dagger a_13^dagger a_15 a_12 + -0.04870772 a_15^dagger a_12^dagger a_14 a_13


In [13]:
# Cell 2 alpha: Faster H2 / molecular fermionic noncommutation graph
#
# Main idea:
#   1. Use cheap fermionic index rules first.
#   2. Only if rules cannot decide, use exact OpenFermion symbolic commutator.
#   3. Never build sparse/dense matrices.
#
# This cell assumes Cell 1 already defined:
#   hermitian_terms
#   operator_to_string

import time
from collections import Counter

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.utils import commutator
from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Fermionic key utilities
# ------------------------------------------------------------

def key_modes(key):
    """
    Modes appearing in one OpenFermion monomial key.

    Example:
        ((3, 1), (0, 1), (3, 0), (0, 0)) -> {0, 3}
    """
    return frozenset(mode for mode, action in key)


def key_creations(key):
    """
    Creation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 1)


def key_annihilations(key):
    """
    Annihilation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 0)


def key_net_delta(key):
    """
    Net occupation change caused by a monomial.

    creation contributes +1
    annihilation contributes -1

    For example:
        a_2^dagger a_1^dagger a_3 a_0
    has delta:
        +1 on modes 2 and 1
        -1 on modes 3 and 0
    """
    delta = Counter()

    for mode, action in key:
        if action == 1:
            delta[mode] += 1
        else:
            delta[mode] -= 1

    return delta


def is_diagonal_key(key):
    """
    True if a monomial preserves occupation mode-by-mode.

    Examples:
        a_p^dagger a_p is diagonal.
        a_p^dagger a_q^dagger a_q a_p is diagonal.
        a_p^dagger a_q is not diagonal when p != q.
    """
    delta = key_net_delta(key)
    return all(value == 0 for value in delta.values())


def is_even_key(key):
    """
    Electronic Hamiltonian terms normally have even fermionic parity:
    length 0, 2, or 4.
    """
    return len(key) % 2 == 0


# ------------------------------------------------------------
# Safe monomial-level commutation rules
# ------------------------------------------------------------

def diagonal_key_commutes_with_key(diagonal_key, other_key):
    """
    Safe rule:

    A diagonal occupation operator depending on modes S commutes with another
    monomial if the other monomial has zero net occupation change on every
    mode in S.
    """
    support = key_modes(diagonal_key)
    delta = key_net_delta(other_key)

    return all(delta.get(mode, 0) == 0 for mode in support)


def no_cross_contractions_even_commute(key_a, key_b):
    """
    Safe rule for normal-ordered even fermionic monomials.

    If there are no possible cross contractions:
        annihilations(A) intersect creations(B) = empty
        annihilations(B) intersect creations(A) = empty

    then even monomials commute.

    This catches many cases beyond completely disjoint support.
    """
    if not is_even_key(key_a) or not is_even_key(key_b):
        return False

    a_ann = key_annihilations(key_a)
    a_cre = key_creations(key_a)

    b_ann = key_annihilations(key_b)
    b_cre = key_creations(key_b)

    return a_ann.isdisjoint(b_cre) and b_ann.isdisjoint(a_cre)


def monomial_pair_definitely_commutes(key_a, key_b):
    """
    Return (True, reason) only when we are sure two monomials commute.
    Return (False, None) if the rule cannot decide.

    Important:
        False here does NOT mean noncommuting.
        It only means "unknown; use exact symbolic fallback."
    """
    # Identity commutes with everything.
    if key_a == () or key_b == ():
        return True, "identity"

    # Any monomial commutes with itself.
    if key_a == key_b:
        return True, "same_monomial"

    # Diagonal occupation-like monomials commute with each other.
    if is_diagonal_key(key_a) and is_diagonal_key(key_b):
        return True, "diagonal_diagonal"

    # Diagonal with excitation-like term, if excitation preserves diagonal support.
    if is_diagonal_key(key_a) and diagonal_key_commutes_with_key(key_a, key_b):
        return True, "diagonal_support_preserved"

    if is_diagonal_key(key_b) and diagonal_key_commutes_with_key(key_b, key_a):
        return True, "diagonal_support_preserved"

    # Even monomials with no cross contractions commute.
    if no_cross_contractions_even_commute(key_a, key_b):
        return True, "no_cross_contractions_even"

    return False, None


# ------------------------------------------------------------
# Operator-level metadata and precheck
# ------------------------------------------------------------

def operator_metadata(op):
    """
    Precompute simple structural data for one FermionOperator.
    """
    keys = list(op.terms.keys())

    modes = set()
    for key in keys:
        modes.update(key_modes(key))

    return {
        "is_zero": len(keys) == 0,
        "only_identity": len(keys) == 1 and keys[0] == (),
        "modes": frozenset(modes),
        "is_even": all(is_even_key(key) for key in keys),
        "is_diagonal": all(is_diagonal_key(key) for key in keys),
        "number_of_monomials": len(keys),
    }


def operator_pair_definitely_commutes(A, B, meta_A, meta_B):
    """
    Return (True, reason) only for guaranteed-commuting pairs.
    Return (False, None) when unresolved.
    """
    if meta_A["is_zero"] or meta_B["is_zero"]:
        return True, "zero"

    if meta_A["only_identity"] or meta_B["only_identity"]:
        return True, "identity"

    # Very cheap global rule:
    # disjoint even fermionic operators commute.
    if (
        meta_A["is_even"]
        and meta_B["is_even"]
        and meta_A["modes"].isdisjoint(meta_B["modes"])
    ):
        return True, "disjoint_even_support"

    # Diagonal occupation-like operators commute with each other.
    if meta_A["is_diagonal"] and meta_B["is_diagonal"]:
        return True, "diagonal_diagonal"

    # More detailed but still cheap:
    # if every monomial pair has a safe commuting reason, the sums commute.
    reasons = Counter()

    for key_a in A.terms:
        for key_b in B.terms:
            ok, reason = monomial_pair_definitely_commutes(key_a, key_b)

            if not ok:
                return False, None

            reasons[reason] += 1

    if len(reasons) > 0:
        main_reason = reasons.most_common(1)[0][0]
        return True, f"all_monomial_pairs_{main_reason}"

    return False, None


# ------------------------------------------------------------
# Exact symbolic fallback
# ------------------------------------------------------------

def exact_symbolic_fermionic_commutator(A, B, tol=1e-12):
    """
    Exact symbolic commutator in fermionic algebra.

    This does not build a 2^n matrix.
    """
    C = normal_ordered(commutator(A, B))
    C.compress(abs_tol=tol)
    return C


# ------------------------------------------------------------
# Alpha graph builder
# ------------------------------------------------------------

def build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
):
    """
    Build noncommutation graph using:
        fast safe index rules first,
        exact symbolic OpenFermion fallback only when needed.

    Parameters
    ----------
    hermitian_terms:
        list of FermionOperator terms T_i from Cell 1.

    tol:
        numerical compression tolerance.

    store_commutators:
        False is recommended for large molecules.
        True is useful for H2 debugging, but can be memory-heavy.

    Returns
    -------
    G:
        networkx.Graph

    stats_df:
        pandas.DataFrame with timing and skip counts
    """
    t_start = time.perf_counter()

    G = nx.Graph()
    stats = Counter()

    metadata = [operator_metadata(T) for T in hermitian_terms]

    # Add vertices.
    for i, T in enumerate(hermitian_terms):
        G.add_node(
            i,
            label=f"T_{i}",
            operator=T,
            operator_string=operator_to_string(T),
            number_of_monomials=len(T.terms),
            modes=sorted(metadata[i]["modes"]),
            is_diagonal=metadata[i]["is_diagonal"],
            is_even=metadata[i]["is_even"],
        )

    n = len(hermitian_terms)

    # Pairwise graph construction.
    for i in range(n):
        A = hermitian_terms[i]
        meta_A = metadata[i]

        for j in range(i + 1, n):
            B = hermitian_terms[j]
            meta_B = metadata[j]

            stats["total_pairs"] += 1

            # 1. Fast guaranteed-commuting rules.
            definitely_commutes, reason = operator_pair_definitely_commutes(
                A, B, meta_A, meta_B
            )

            if definitely_commutes:
                stats["pairs_skipped_by_index_rules"] += 1
                stats[f"skip_{reason}"] += 1
                continue

            # 2. Exact symbolic fallback.
            stats["pairs_sent_to_exact_symbolic"] += 1

            Cij = exact_symbolic_fermionic_commutator(A, B, tol=tol)

            if len(Cij.terms) != 0:
                stats["noncommuting_edges"] += 1

                edge_data = {
                    "method": "exact_symbolic_fallback",
                    "meaning": f"[T_{i}, T_{j}] != 0",
                }

                if store_commutators:
                    edge_data["commutator"] = Cij
                    edge_data["commutator_string"] = operator_to_string(Cij)
                else:
                    edge_data["commutator_string"] = (
                        "(not stored; rerun with store_commutators=True)"
                    )

                G.add_edge(i, j, **edge_data)

            else:
                stats["exact_symbolic_found_commuting"] += 1

    elapsed = time.perf_counter() - t_start

    stats["vertices"] = n
    stats["edges"] = G.number_of_edges()
    stats["commuting_pairs"] = stats["total_pairs"] - G.number_of_edges()
    stats["elapsed_seconds"] = elapsed

    stats_df = pd.DataFrame([dict(stats)])

    return G, stats_df


# ------------------------------------------------------------
# Run alpha graph builder
# ------------------------------------------------------------

# For H2 debugging, you can set store_commutators=True.
# For larger molecules, keep this False.
G_alpha, stats_df = build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
)

print("=== Fermionic noncommutation graph alpha summary ===")
display(stats_df)

print("Number of vertices / fermionic terms:", G_alpha.number_of_nodes())
print("Number of noncommuting pairs / edges:", G_alpha.number_of_edges())
print("Is graph bipartite?", nx.is_bipartite(G_alpha))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G_alpha.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G_alpha.degree[i],
            "commutes_with_all": G_alpha.degree[i] == 0,
            "number_of_monomials": data["number_of_monomials"],
            "modes": data["modes"],
            "is_diagonal": data["is_diagonal"],
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df_alpha = pd.DataFrame(vertex_rows)

print("\n=== Alpha vertices: fermionic terms ===")
display(vertex_df_alpha)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G_alpha.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": data["meaning"],
            "method": data["method"],
            "commutator": data["commutator_string"],
        }
    )

edge_df_alpha = pd.DataFrame(edge_rows)

print("\n=== Alpha edges: noncommuting pairs ===")
display(edge_df_alpha)

# The commutation graph is too large for this.

# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G_alpha)

# node_labels = {
#     i: f"T_{i}"
#     for i in G_alpha.nodes()
# }

# node_sizes = [
#     1000 + 250 * G_alpha.degree[i]
#     for i in G_alpha.nodes()
# ]

# nx.draw_networkx_nodes(G_alpha, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G_alpha, pos, width=1.5)
# nx.draw_networkx_labels(
#     G_alpha,
#     pos,
#     labels=node_labels,
#     font_size=11,
#     font_weight="bold",
# )

# plt.title("Fermionic Noncommutation Graph Alpha")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph alpha summary ===


,total_pairs,pairs_skipped_by_index_rules,skip_identity,pairs_sent_to_exact_symbolic,noncommuting_edges,skip_disjoint_even_support,skip_diagonal_diagonal,skip_all_monomial_pairs_diagonal_support_preserved,exact_symbolic_found_commuting,vertices,edges,commuting_pairs,elapsed_seconds
0,2910078,1081006,2412,1829072,1663978,1068218,1920,8456,165094,2413,1663978,1246100,104.862653


Number of vertices / fermionic terms: 2413
Number of noncommuting pairs / edges: 1663978
Is graph bipartite? False

=== Alpha vertices: fermionic terms ===


,vertex,degree,commutes_with_all,number_of_monomials,modes,is_diagonal,fermionic_term
0,T_0,0,True,1,[],True,-22.40891512 I
1,T_1,428,False,1,[0],True,-4.00514475 a_0^dagger a_0
2,T_2,838,False,2,"[0, 14]",False,-0.53295796 a_0^dagger a_14 + -0.53295796 a_14^dagger a_0
3,T_3,428,False,1,[1],True,-4.00514475 a_1^dagger a_1
4,T_4,838,False,2,"[1, 15]",False,-0.53295796 a_1^dagger a_15 + -0.53295796 a_15^dagger a_1
...,...,...,...,...,...,...,...
2408,T_2408,1424,False,2,"[12, 13, 14, 15]",False,+0.10185535 a_14^dagger a_13^dagger a_15 a_12 + +0.10185535 a_15^dagger a_12^dagger a_14 a_13
2409,T_2409,817,False,1,"[12, 15]",True,-0.47820478 a_15^dagger a_12^dagger a_15 a_12
2410,T_2410,817,False,1,"[13, 14]",True,-0.47820478 a_14^dagger a_13^dagger a_14 a_13
2411,T_2411,785,False,1,"[13, 15]",True,-0.37634943 a_15^dagger a_13^dagger a_15 a_13



=== Alpha edges: noncommuting pairs ===


,source,target,meaning,method,commutator
0,T_1,T_2,"[T_1, T_2] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1,T_1,T_38,"[T_1, T_38] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
2,T_1,T_39,"[T_1, T_39] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3,T_1,T_40,"[T_1, T_40] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
4,T_1,T_41,"[T_1, T_41] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
...,...,...,...,...,...
1663973,T_2405,T_2410,"[T_2405, T_2410] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1663974,T_2405,T_2412,"[T_2405, T_2412] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1663975,T_2406,T_2412,"[T_2406, T_2412] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1663976,T_2408,T_2409,"[T_2408, T_2409] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)


In [14]:
print("Same edge set?")
print(set(G.edges()) == set(G_alpha.edges()))

Same edge set?
True


In [15]:
# Cell 3: Color graph, build commuting blocks, map JW/BK, verify commutation

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from itertools import combinations
from openfermion.ops import FermionOperator
from openfermion.transforms import normal_ordered, jordan_wigner, bravyi_kitaev

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# 1. Color the noncommutation graph
# ------------------------------------------------------------
# Since edges mean noncommutation, each color class is a commuting group.

coloring = nx.coloring.greedy_color(G, strategy="largest_first")

color_groups = {}

for node, color in coloring.items():
    color_groups.setdefault(color, []).append(node)

for color in color_groups:
    color_groups[color] = sorted(color_groups[color])

color_names = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "brown",
    "pink",
    "gray",
]

color_name = {
    color: color_names[color] if color < len(color_names) else f"color_{color}"
    for color in color_groups
}

num_grouped_terms = sum(len(nodes) for nodes in color_groups.values())

print("Number of fermionic terms / vertices:", G.number_of_nodes())
print("Number of colors / commuting groups:", len(color_groups))
print("Number of grouped terms:", num_grouped_terms)

assert num_grouped_terms == G.number_of_nodes()


# ------------------------------------------------------------
# 2. Verify each color group is mutually commuting
# ------------------------------------------------------------

def verify_commuting_group(nodes, tol=1e-12):
    for i, j in combinations(nodes, 2):
        A = G.nodes[i]["operator"]
        B = G.nodes[j]["operator"]

        if not commute(A, B, tol=tol):
            return False

    return True


group_summary_rows = []

for color, nodes in sorted(color_groups.items()):
    group_summary_rows.append(
        {
            "color_id": color,
            "color_name": color_name[color],
            "number_of_terms": len(nodes),
            "vertices": [f"T_{i}" for i in nodes],
            "verified_mutually_commuting": verify_commuting_group(nodes),
        }
    )

group_summary_df = pd.DataFrame(group_summary_rows)

print("\n=== Commuting groups from graph coloring ===")
display(group_summary_df)


# ------------------------------------------------------------
# 3. Build Hamiltonian pieces by color
# ------------------------------------------------------------

H_by_color = {}

for color, nodes in sorted(color_groups.items()):
    H_color = FermionOperator.zero()

    for node in nodes:
        H_color += G.nodes[node]["operator"]

    H_color = normal_ordered(H_color)
    H_color.compress(abs_tol=1e-12)

    H_by_color[color] = H_color


color_block_rows = []

for color, nodes in sorted(color_groups.items()):
    for local_index, node in enumerate(nodes, start=1):
        color_block_rows.append(
            {
                "color_block": f"H_{color_name[color]}",
                "local_term_name": f"{color_name[color][0].upper()}_{local_index}",
                "vertex": f"T_{node}",
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

color_block_df = pd.DataFrame(color_block_rows)

print("\n=== Which T_i belongs to which color block ===")
display(color_block_df)


print("\n=== Hamiltonian split by commuting color groups ===")

for color, H_color in H_by_color.items():
    nodes = color_groups[color]

    print("\n" + "=" * 80)
    print(f"H_{color_name[color]} consists of:")
    print(" + ".join([f"T_{node}" for node in nodes]))

    print(f"\nSummed operator H_{color_name[color]} =")
    print(H_color)


# ------------------------------------------------------------
# 4. Trotter ordering induced by color groups
# ------------------------------------------------------------

trotter_order = []

for color, nodes in sorted(color_groups.items()):
    for node in nodes:
        trotter_order.append(node)

print("\n=== Trotter order by commuting color groups ===")
print([f"T_{i}" for i in trotter_order])


# ------------------------------------------------------------
# 5. Helper functions for JW/BK output
# ------------------------------------------------------------

def sort_qubit_key(term):
    return (len(term), term)


def format_qubit_term(term):
    if term == ():
        return "I"

    return " ".join([f"{pauli}{qubit}" for qubit, pauli in term])


def qubit_coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def qubit_operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_qubit_key(item[0])):
        pieces.append(f"{qubit_coeff_to_str(coeff, digits)} {format_qubit_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def apply_bk(op, n_qubits):
    try:
        return bravyi_kitaev(op, n_qubits=n_qubits)
    except TypeError:
        return bravyi_kitaev(op, n_qubits)


def qubit_commutator(A, B, tol=1e-12):
    C = A * B - B * A
    C.compress(abs_tol=tol)
    return C


def qubit_commute(A, B, tol=1e-12):
    C = qubit_commutator(A, B, tol=tol)
    return len(C.terms) == 0


n_qubits = molecule.n_qubits

print("\nNumber of qubits / spin orbitals:", n_qubits)


# ------------------------------------------------------------
# 6. Map each fermionic vertex T_i to JW(T_i) and BK(T_i)
# ------------------------------------------------------------

mapped_rows = []

for node in sorted(G.nodes()):
    T_i = G.nodes[node]["operator"]

    JW_T_i = jordan_wigner(T_i)
    JW_T_i.compress(abs_tol=1e-12)

    BK_T_i = apply_bk(T_i, n_qubits=n_qubits)
    BK_T_i.compress(abs_tol=1e-12)

    G.nodes[node]["JW_operator"] = JW_T_i
    G.nodes[node]["BK_operator"] = BK_T_i
    G.nodes[node]["JW_operator_string"] = qubit_operator_to_string(JW_T_i)
    G.nodes[node]["BK_operator_string"] = qubit_operator_to_string(BK_T_i)

    node_color = coloring[node]
    node_color_name = color_name[node_color]

    mapped_rows.append(
        {
            "color": node_color_name,
            "vertex": f"T_{node}",
            "fermionic_term": G.nodes[node]["operator_string"],
            "number_of_JW_Pauli_strings": len(JW_T_i.terms),
            "JW_transform": qubit_operator_to_string(JW_T_i),
            "number_of_BK_Pauli_strings": len(BK_T_i.terms),
            "BK_transform": qubit_operator_to_string(BK_T_i),
        }
    )

mapped_terms_df = pd.DataFrame(mapped_rows)

print("\n=== Fermionic terms mapped to JW and BK ===")
display(mapped_terms_df)


# ------------------------------------------------------------
# 7. Map each color block H_color to JW and BK
# ------------------------------------------------------------

JW_by_color = {}
BK_by_color = {}

block_rows = []

for color, H_color in sorted(H_by_color.items()):
    JW_color = jordan_wigner(H_color)
    JW_color.compress(abs_tol=1e-12)

    BK_color = apply_bk(H_color, n_qubits=n_qubits)
    BK_color.compress(abs_tol=1e-12)

    JW_by_color[color] = JW_color
    BK_by_color[color] = BK_color

    block_rows.append(
        {
            "color_block": f"H_{color_name[color]}",
            "fermionic_vertices": " + ".join([f"T_{node}" for node in color_groups[color]]),
            "number_of_fermionic_terms": len(color_groups[color]),
            "number_of_JW_Pauli_strings": len(JW_color.terms),
            "JW_block": qubit_operator_to_string(JW_color),
            "number_of_BK_Pauli_strings": len(BK_color.terms),
            "BK_block": qubit_operator_to_string(BK_color),
        }
    )

block_map_df = pd.DataFrame(block_rows)

print("\n=== Color blocks mapped to JW and BK ===")
display(block_map_df)


# ------------------------------------------------------------
# 8. Verify same-color terms commute after JW and BK
# ------------------------------------------------------------

verification_rows = []

for color, nodes in sorted(color_groups.items()):
    for i, j in combinations(nodes, 2):
        Ti = G.nodes[i]["operator"]
        Tj = G.nodes[j]["operator"]

        JW_Ti = G.nodes[i]["JW_operator"]
        JW_Tj = G.nodes[j]["JW_operator"]

        BK_Ti = G.nodes[i]["BK_operator"]
        BK_Tj = G.nodes[j]["BK_operator"]

        verification_rows.append(
            {
                "color_group": color_name[color],
                "pair": f"T_{i}, T_{j}",
                "fermionic_commute": commute(Ti, Tj),
                "JW_commute": qubit_commute(JW_Ti, JW_Tj),
                "BK_commute": qubit_commute(BK_Ti, BK_Tj),
            }
        )

verification_df = pd.DataFrame(verification_rows)

print("\n=== Verify commuting groups after JW/BK mapping ===")
display(verification_df)

Number of fermionic terms / vertices: 2413
Number of colors / commuting groups: 267
Number of grouped terms: 2413

=== Commuting groups from graph coloring ===


,color_id,color_name,number_of_terms,vertices,verified_mutually_commuting
0,0,red,13,"[T_0, T_64, T_466, T_471, T_905, T_915, T_966, T_1020, T_1347, T_1679, T_1797, T_2295, T_2307]",True
1,1,blue,12,"[T_2, T_4, T_470, T_901, T_921, T_962, T_1022, T_1398, T_1878, T_1980, T_2298, T_2321]",True
2,2,green,16,"[T_1, T_35, T_63, T_461, T_895, T_897, T_955, T_979, T_1028, T_1380, T_1520, T_1895, T_2135, T_2305, T_2314, T_2324]",True
3,3,orange,10,"[T_463, T_903, T_1045, T_1100, T_1307, T_1349, T_1350, T_1651, T_2315, T_2325]",True
4,4,purple,10,"[T_465, T_899, T_980, T_1021, T_1029, T_1318, T_1404, T_1893, T_2348, T_2355]",True
...,...,...,...,...,...
262,262,color_262,6,"[T_619, T_784, T_1543, T_1551, T_1780, T_1816]",True
263,263,color_263,3,"[T_1362, T_1960, T_2374]",True
264,264,color_264,2,"[T_1976, T_2292]",True
265,265,color_265,7,"[T_553, T_621, T_689, T_751, T_810, T_869, T_2404]",True



=== Which T_i belongs to which color block ===


,color_block,local_term_name,vertex,fermionic_term
0,H_red,R_1,T_0,-22.40891512 I
1,H_red,R_2,T_64,-0.07342357 a_1^dagger a_0^dagger a_15 a_14 + -0.07342357 a_15^dagger a_14^dagger a_1 a_0
2,H_red,R_3,T_466,+0.04316516 a_10^dagger a_4^dagger a_14 a_0 + +0.04316516 a_14^dagger a_0^dagger a_10 a_4
3,H_red,R_4,T_471,+0.07342357 a_14^dagger a_1^dagger a_15 a_0 + +0.07342357 a_15^dagger a_0^dagger a_14 a_1
4,H_red,R_5,T_905,-0.00551298 a_13^dagger a_7^dagger a_15 a_1 + -0.00551298 a_15^dagger a_1^dagger a_13 a_7
...,...,...,...,...
2408,H_color_265,C_4,T_751,+0.07289402 a_9^dagger a_1^dagger a_15 a_9 + +0.07289402 a_15^dagger a_9^dagger a_9 a_1
2409,H_color_265,C_5,T_810,+0.07289402 a_11^dagger a_1^dagger a_15 a_11 + +0.07289402 a_15^dagger a_11^dagger a_11 a_1
2410,H_color_265,C_6,T_869,+0.07289402 a_13^dagger a_1^dagger a_15 a_13 + +0.07289402 a_15^dagger a_13^dagger a_13 a_1
2411,H_color_265,C_7,T_2404,-0.01913183 a_13^dagger a_12^dagger a_14 a_13 + -0.01913183 a_14^dagger a_13^dagger a_13 a_12



=== Hamiltonian split by commuting color groups ===

H_red consists of:
T_0 + T_64 + T_466 + T_471 + T_905 + T_915 + T_966 + T_1020 + T_1347 + T_1679 + T_1797 + T_2295 + T_2307

Summed operator H_red =
-22.408915115183795 [] +
-0.07342356567648453 [1^ 0^ 15 14] +
0.02207805822580939 [3^ 2^ 6 5] +
0.02207805822580939 [5^ 2^ 6 3] +
0.0020192893932280765 [5^ 3^ 11 9] +
-0.015810495673025112 [6^ 2^ 12 8] +
0.02207805822580939 [6^ 3^ 5 2] +
0.02207805822580939 [6^ 5^ 3 2] +
-0.0005647002437223717 [7^ 4^ 13 10] +
0.005528888690091464 [9^ 8^ 12 11] +
0.04316515655384267 [10^ 4^ 14 0] +
0.0005647002437223717 [10^ 7^ 13 4] +
0.005528888690091464 [11^ 8^ 12 9] +
0.0020192893932280765 [11^ 9^ 5 3] +
-0.015810495673025112 [12^ 8^ 6 2] +
0.005528888690091464 [12^ 9^ 11 8] +
0.005528888690091464 [12^ 11^ 9 8] +
0.0005647002437223717 [13^ 4^ 10 7] +
-0.005512979769186424 [13^ 7^ 15 1] +
-0.0005647002437223717 [13^ 10^ 7 4] +
0.043165156553842676 [14^ 0^ 10 4] +
0.07342356567648453 [14^ 1^ 15 0] +
0.

,color,vertex,fermionic_term,number_of_JW_Pauli_strings,JW_transform,number_of_BK_Pauli_strings,BK_transform
0,red,T_0,-22.40891512 I,1,-22.40891512 I,1,-22.40891512 I
1,green,T_1,-4.00514475 a_0^dagger a_0,2,-2.00257238 I + +2.00257238 Z0,2,-2.00257238 I + +2.00257238 Z0
2,blue,T_2,-0.53295796 a_0^dagger a_14 + -0.53295796 a_14^dagger a_0,2,-0.26647898 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 X14 + -0.26647898 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Y14,2,-0.26647898 X0 X1 X3 Y7 Z11 Z13 Y14 + +0.26647898 Y0 X1 X3 Y7 Z11 Z13 X14
3,brown,T_3,-4.00514475 a_1^dagger a_1,2,-2.00257238 I + +2.00257238 Z1,2,-2.00257238 I + +2.00257238 Z0 Z1
4,blue,T_4,-0.53295796 a_1^dagger a_15 + -0.53295796 a_15^dagger a_1,2,-0.26647898 X1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 X15 + -0.26647898 Y1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Y15,2,+0.26647898 Z0 X1 X3 X7 Z15 + +0.26647898 Y1 X3 Y7 Z11 Z13 Z14
...,...,...,...,...,...,...,...
2408,color_86,T_2408,+0.10185535 a_14^dagger a_13^dagger a_15 a_12 + +0.10185535 a_15^dagger a_12^dagger a_14 a_13,8,-0.01273192 X12 X13 X14 X15 + -0.01273192 X12 X13 Y14 Y15 + -0.01273192 X12 Y13 X14 Y15 + +0.01273192 X12 Y13 Y14 X15 + +0.01273192 Y12 X13 X14 Y15 + -0.01273192 Y12 X13 Y14 X15 + -0.01273192 Y12 Y13 X14 X15 + -0.01273192 Y12 Y13 Y14 Y15,8,-0.01273192 X12 X14 + -0.01273192 Y12 Y14 + +0.01273192 X12 Z13 X14 + +0.01273192 Y12 Z13 Y14 + -0.01273192 Z7 Z11 X12 X14 Z15 + -0.01273192 Z7 Z11 Y12 Y14 Z15 + +0.01273192 Z7 Z11 X12 Z13 X14 Z15 + +0.01273192 Z7 Z11 Y12 Z13 Y14 Z15
2409,color_18,T_2409,-0.47820478 a_15^dagger a_12^dagger a_15 a_12,4,+0.11955119 I + -0.11955119 Z12 + -0.11955119 Z15 + +0.11955119 Z12 Z15,4,+0.11955119 I + -0.11955119 Z12 + -0.11955119 Z7 Z11 Z13 Z14 Z15 + +0.11955119 Z7 Z11 Z12 Z13 Z14 Z15
2410,color_24,T_2410,-0.47820478 a_14^dagger a_13^dagger a_14 a_13,4,+0.11955119 I + -0.11955119 Z13 + -0.11955119 Z14 + +0.11955119 Z13 Z14,4,+0.11955119 I + -0.11955119 Z14 + -0.11955119 Z12 Z13 + +0.11955119 Z12 Z13 Z14
2411,color_23,T_2411,-0.37634943 a_15^dagger a_13^dagger a_15 a_13,4,+0.09408736 I + -0.09408736 Z13 + -0.09408736 Z15 + +0.09408736 Z13 Z15,4,+0.09408736 I + -0.09408736 Z12 Z13 + +0.09408736 Z7 Z11 Z12 Z14 Z15 + -0.09408736 Z7 Z11 Z13 Z14 Z15



=== Color blocks mapped to JW and BK ===


,color_block,fermionic_vertices,number_of_fermionic_terms,number_of_JW_Pauli_strings,JW_block,number_of_BK_Pauli_strings,BK_block
0,H_red,T_0 + T_64 + T_466 + T_471 + T_905 + T_915 + T_966 + T_1020 + T_1347 + T_1679 + T_1797 + T_2295 + T_2307,13,49,-22.40891512 I + -0.01835589 X0 X1 Y14 Y15 + +0.01835589 X0 Y1 Y14 X15 + +0.01835589 Y0 X1 X14 Y15 + -0.01835589 Y0 Y1 X14 X15 + -0.00551951 X2 X3 X5 X6 + -0.00551951 X2 Y3 Y5 X6 + -0.00551951 Y2 X3 X5 Y6 + -0.00551951 Y2 Y3 Y5 Y6 + -0.00138222 X8 X9 X11 X12 + -0.00138222 X8 Y9 Y11 X12 + -0.00138222 Y8 X9 X11 Y12 + -0.00138222 Y8 Y9 Y11 Y12 + -0.00025241 X3 Z4 X5 X9 Z10 X11 + +0.00025241 X3 Z4 X5 Y9 Z10 Y11 + -0.00025241 X3 Z4 Y5 X9 Z10 Y11 + -0.00025241 X3 Z4 Y5 Y9 Z10 X11 + -0.00025241 Y3 Z4 X5 X9 Z10 Y11 + -0.00025241 Y3 Z4 X5 Y9 Z10 X11 + +0.00025241 Y3 Z4 Y5 X9 Z10 X11 + -0.00025241 Y3 Z4 Y5 Y9 Z10 Y11 + -0.00014118 X4 Z5 Z6 X7 Y10 Z11 Z12 Y13 + +0.00014118 X4 Z5 Z6 Y7 Y10 Z11 Z12 X13 + +0.00014118 Y4 Z5 Z6 X7 X10 Z11 Z12 Y13 + -0.00014118 Y4 Z5 Z6 Y7 X10 Z11 Z12 X13 + -0.00539564 X0 Z1 Z2 Z3 X4 X10 Z11 Z12 Z13 X14 + -0.00539564 X0 Z1 Z2 Z3 X4 Y10 Z11 Z12 Z13 Y14 + -0.00539564 X0 Z1 Z2 Z3 Y4 X10 Z11 Z12 Z13 Y14 + +0.00539564 X0 Z1 Z2 Z3 Y4 Y10 Z11 Z12 Z13 X14 + +0.00539564 Y0 Z1 Z2 Z3 X4 X10 Z11 Z12 Z13 Y14 + -0.00539564 Y0 Z1 Z2 Z3 X4 Y10 Z11 Z12 Z13 X14 + -0.00539564 Y0 Z1 Z2 Z3 Y4 X10 Z11 Z12 Z13 X14 + -0.00539564 Y0 Z1 Z2 Z3 Y4 Y10 Z11 Z12 Z13 Y14 + +0.00068912 X1 Z2 Z3 Z4 Z5 Z6 X7 X13 Z14 X15 + +0.00068912 X1 Z2 Z3 Z4 Z5 Z6 X7 Y13 Z14 Y15 + +0.00068912 X1 Z2 Z3 Z4 Z5 Z6 Y7 X13 Z14 Y15 + -0.00068912 X1 Z2 Z3 Z4 Z5 Z6 Y7 Y13 Z14 X15 + -0.00068912 Y1 Z2 Z3 Z4 Z5 Z6 X7 X13 Z14 Y15 + +0.00068912 Y1 Z2 Z3 Z4 Z5 Z6 X7 Y13 Z14 X15 + +0.00068912 Y1 Z2 Z3 Z4 Z5 Z6 Y7 X13 Z14 X15 + +0.00068912 Y1 Z2 Z3 Z4 Z5 Z6 Y7 Y13 Z14 Y15 + +0.00197631 X2 Z3 Z4 Z5 X6 X8 Z9 Z10 Z11 X12 + -0.00197631 X2 Z3 Z4 Z5 X6 Y8 Z9 Z10 Z11 Y12 + +0.00197631 X2 Z3 Z4 Z5 Y6 X8 Z9 Z10 Z11 Y12 + +0.00197631 X2 Z3 Z4 Z5 Y6 Y8 Z9 Z10 Z11 X12 + +0.00197631 Y2 Z3 Z4 Z5 X6 X8 Z9 Z10 Z11 Y12 + +0.00197631 Y2 Z3 Z4 Z5 X6 Y8 Z9 Z10 Z11 X12 + -0.00197631 Y2 Z3 Z4 Z5 Y6 X8 Z9 Z10 Z11 X12 + +0.00197631 Y2 Z3 Z4 Z5 Y6 Y8 Z9 Z10 Z11 Y12,49,-22.40891512 I + +0.01835589 X0 Z1 X14 + +0.01835589 Y0 Z1 Y14 + -0.00551951 X2 X5 X6 + -0.00551951 Y2 X5 Y6 + -0.00025241 X3 Y5 Y9 Z11 + -0.00138222 X8 X11 X12 X13 + -0.00138222 Y8 X11 Y12 X13 + +0.00068912 Y1 X3 Z11 Y13 Z15 + -0.00025241 X3 Z4 X5 X9 Z10 + -0.00025241 X3 Y5 Z8 Y9 Z10 + +0.00138222 X8 Z10 Y11 Y12 X13 + -0.00138222 Y8 Z10 Y11 X12 X13 + +0.01835589 X0 Z7 Z11 Z13 X14 Z15 + +0.01835589 Y0 Z7 Z11 Z13 Y14 Z15 + -0.00068912 Z0 X1 X3 Z7 X13 Z14 + -0.00068912 Y1 X3 Z7 Z12 Y13 Z14 + -0.00068912 Y1 Y3 Z5 Z6 X13 Z14 + +0.00551951 Z1 X2 Z3 Z4 Y5 Y6 + -0.00551951 Z1 Y2 Z3 Z4 Y5 X6 + +0.00025241 Z1 Z2 Y3 Y5 X9 Z10 + -0.00025241 X3 Z4 X5 Z8 X9 Z11 + +0.00068912 Z0 X1 X3 Z11 Z12 X13 Z15 + -0.00025241 Z1 Z2 Y3 Z4 X5 Y9 Z11 + +0.00025241 Z1 Z2 Y3 Y5 Z8 X9 Z11 + -0.00014118 X4 Y5 Z6 Z9 Y10 Y11 Y13 + +0.00014118 Y4 Y5 Z6 Z9 X10 Y11 Y13 + +0.00068912 Z0 X1 Y3 Z5 Z6 Z12 Y13 Z14 + -0.00025241 Z1 Z2 Y3 Z4 X5 Z8 Y9 Z10 + -0.00068912 Z0 X1 Y3 Z5 Z6 Z7 Z11 Y13 Z15 + +0.00068912 Y1 Y3 Z5 Z6 Z7 Z11 Z12 X13 Z15 + -0.00014118 Z3 X4 X5 Z7 Z9 Y10 Y11 Z12 X13 + +0.00014118 Z3 Y4 X5 Z7 Z9 X10 Y11 Z12 X13 + -0.00539564 X0 X1 Y3 X4 X5 Z9 X10 Y11 Z13 X14 + -0.00539564 X0 X1 Y3 X4 X5 Z9 Y10 Y11 Z13 Y14 + -0.00539564 X0 X1 Y3 Y4 X5 Z9 X10 Y11 Z13 Y14 + +0.00539564 X0 X1 Y3 Y4 X5 Z9 Y10 Y11 Z13 X14 + +0.00539564 Y0 X1 Y3 X4 X5 Z9 X10 Y11 Z13 Y14 + -0.00539564 Y0 X1 Y3 X4 X5 Z9 Y10 Y11 Z13 X14 + -0.00539564 Y0 X1 Y3 Y4 X5 Z9 X10 Y11 Z13 X14 + -0.00539564 Y0 X1 Y3 Y4 X5 Z9 Y10 Y11 Z13 Y14 + +0.00197631 Z1 X2 Y3 Z5 X6 X8 X9 Y11 X12 X13 + -0.00197631 Z1 X2 Y3 Z5 X6 Y8 X9 Y11 Y12 X13 + +0.00197631 Z1 X2 Y3 Z5 Y6 X8 X9 Y11 Y12 X13 + +0.00197631 Z1 X2 Y3 Z5 Y6 Y8 X9 Y11 X12 X13 + +0.00197631 Z1 Y2 Y3 Z5 X6 X8 X9 Y11 Y12 X13 + +0.00197631 Z1 Y2 Y3 Z5 X6 Y8 X9 Y11 X12 X13 + -0.00197631 Z1 Y2 Y3 Z5 Y6 X8 X9 Y11 X12 X13 + +0.


=== Verify commuting groups after JW/BK mapping ===


,color_group,pair,fermionic_commute,JW_commute,BK_commute
0,red,"T_0, T_64",True,True,True
1,red,"T_0, T_466",True,True,True
2,red,"T_0, T_471",True,True,True
3,red,"T_0, T_905",True,True,True
4,red,"T_0, T_915",True,True,True
...,...,...,...,...,...
10588,color_265,"T_751, T_869",True,True,True
10589,color_265,"T_751, T_2404",True,True,True
10590,color_265,"T_810, T_869",True,True,True
10591,color_265,"T_810, T_2404",True,True,True


In [16]:
# Find duplicated JW Pauli strings across fermionic terms T_i

from collections import defaultdict
import pandas as pd

pauli_usage = defaultdict(list)

for node in sorted(G.nodes()):
    JW_T = G.nodes[node]["JW_operator"]

    for pauli_key, coeff in JW_T.terms.items():
        pauli_string = format_qubit_term(pauli_key)

        pauli_usage[pauli_string].append(
            {
                "vertex": f"T_{node}",
                "coefficient": coeff,
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

duplicate_rows = []

for pauli_string, appearances in pauli_usage.items():
    if len(appearances) > 1:
        duplicate_rows.append(
            {
                "JW_Pauli_string": pauli_string,
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "coefficients": [x["coefficient"] for x in appearances],
            }
        )

duplicate_jw_df = pd.DataFrame(duplicate_rows)
duplicate_jw_df = duplicate_jw_df.sort_values(
    "number_of_appearances",
    ascending=False
).reset_index(drop=True)

print("Total JW Pauli-string appearances:", sum(len(G.nodes[node]["JW_operator"].terms) for node in G.nodes()))
print("Number of unique JW Pauli strings:", len(pauli_usage))
print("Number of duplicated JW Pauli strings:", len(duplicate_jw_df))

display(duplicate_jw_df)

Total JW Pauli-string appearances: 16185
Number of unique JW Pauli strings: 7921
Number of duplicated JW Pauli strings: 6593


,JW_Pauli_string,number_of_appearances,appears_in_vertices,coefficients
0,I,137,"[T_0, T_1, T_3, T_5, T_9, T_13, T_17, T_21, T_25, T_29, T_30, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_86, T_134, T_161, T_203, T_236, T_272, T_311, T_335, T_380, T_398, T_449, T_461, T_496, T_498, T_546, T_569, T_611, T_640, T_676, T_714, T_738, T_779, T_797, T_844, T_856, T_894, T_896, T_906, T_955, T_971, T_1012, T_1034, T_1068, T_1103, T_1131, T_1171, T_1192, T_1237, T_1251, T_1295, T_1299, T_1340, T_1356, T_1390, T_1412, T_1440, T_1475, T_1496, T_1536, T_1550, T_1593, T_1597, T_1607, T_1643, T_1658, T_1687, T_1710, T_1734, T_1761, T_1779, T_1810, T_1822, T_1853, T_1857, T_1886, T_1901, T_1925, T_1948, T_1966, T_1993, T_2005, T_2035, T_2039, T_2046, T_2071, T_2084, T_2104, T_2120, T_2135, T_2154, T_2164, ...]","[-22.408915115183795, -2.0025723751678735, -2.0025723751678735, -1.8076608544243982, -1.8076608544243982, -1.8076608544243995, -1.8076608544243995, -1.8076608544243968, -1.8076608544243968, -1.4620212328253892, -1.4620212328253892, -1.462021232825395, -1.462021232825395, -1.4620212328254003, -1.4620212328254003, -1.367450382110078, -1.367450382110078, 0.12601084193294032, 0.09441778014524217, 0.12150482690042828, 0.09441778014524223, 0.1215048269004284, 0.09441778014524212, 0.1215048269004282, 0.11151132357155509, 0.12341729243332238, 0.11151132357155553, 0.12341729243332288, 0.11151132357155592, 0.1234172924333233, 0.09931008213958181, 0.11766597355870295, 0.12150482690042828, 0.09441778014524217, 0.1215048269004284, 0.09441778014524223, 0.1215048269004282, 0.09441778014524212, 0.12341729243332238, 0.11151132357155509, 0.12341729243332288, 0.11151132357155553, 0.1234172924333233, 0.11151132357155592, 0.11766597355870295, 0.09931008213958181, 0.13450436113594946, 0.10134886265737106, 0.11170989829963887, 0.1013488626573709, 0.10926625002654566, 0.1064870678331826, 0.1188088700377998, 0.10569378505039045, 0.1095313045443218, 0.10275103248778099, 0.13172862963369106, 0.10338461264647633, 0.11511605099515307, 0.11170989829963887, 0.10134886265737106, 0.10926625002654566, 0.1013488626573709, 0.1188088700377998, 0.1064870678331826, 0.1095313045443218, 0.10569378505039045, 0.13172862963369106, 0.10275103248778099, 0.11511605099515307, 0.10338461264647633, 0.13581597072077464, 0.10134886265737097, 0.10795464044172066, 0.10445999419292423, 0.11562857121526804, 0.10434289120617345, 0.12895196049698127, 0.10612899997225601, 0.11548827250356242, 0.10338461264647705, 0.11511605099515433, 0.10795464044172066, 0.10134886265737097, 0.11562857121526804, 0.10445999419292423, 0.12895196049698127, 0.10434289120617345, 0.11548827250356242, 0.10612899997225601, 0.11511605099515433, 0.10338461264647705, 0.1382596189938675, 0.10398482334524581, 0.12563136296274302, 0.10489520911479004, 0.12158553917450926, 0.10605185291131804, 0.11285190207856013, 0.10338461264647632, ...]"
1,Z3,16,"[T_9, T_86, T_546, T_906, T_1299, T_1340, T_1356, T_1390, T_1412, T_1440, T_1475, T_1496, T_1536, T_1550, T_1593, T_1597]","[1.8076608544243982, -0.12150482690042828, -0.09441778014524217, -0.13450436113594946, -0.11170989829963887, -0.10134886265737106, -0.10926625002654566, -0.1013488626573709, -0.1188088700377998, -0.1064870678331826, -0.1095313045443218, -0.10569378505039045, -0.13172862963369106, -0.10275103248778099, -0.11511605099515307, -0.10338461264647633]"
2,Z0,16,"[T_1, T_37, T_65, T_86, T_134, T_161, T_203, T_236, T_272, T_311, T_335, T_380, T_398, T_449, T_461, T_496]","[2.0025723751678735, -0.12601084193294032, -0.09441778014524217, -0.12150482690042828, -0.09441778014524223, -0.1215048269004284, -0.09441778014524212, -0.1215048269004282, -0.11151132357155509, -0.12341729243332238, -0.11151132357155553, -0.12341729243332288, -0.11151132357155592, -0.1234172924333233, -0.09931008213958181, -0.11766597355870295]"
3,Z15,16,"[T_36, T_496, T_896, T_1295, T_1597, T_1853, T_2039, T_2184, T_2285, T_2342, T_2371, T_2393, T_2402, T_2409, T_2411, T_2412]","[1.36745038

In [17]:
# Compute Pauli-duplication ratio for JW and BK

from collections import defaultdict
import pandas as pd

from openfermion.ops import FermionOperator
from openfermion.transforms import jordan_wigner, bravyi_kitaev, normal_ordered


def pauli_support(qubit_op, tol=1e-12, include_identity=True):
    """
    Return the set of Pauli strings with nonzero coefficients.
    """
    qubit_op.compress(abs_tol=tol)

    support = set()

    for pauli_key, coeff in qubit_op.terms.items():
        if abs(coeff) <= tol:
            continue

        if not include_identity and pauli_key == ():
            continue

        support.add(pauli_key)

    return support


def map_fermion_to_qubit(op, mapping="JW", n_qubits=None):
    """
    Map a FermionOperator to a QubitOperator using JW or BK.
    """
    mapping = mapping.upper()

    if mapping == "JW":
        qop = jordan_wigner(op)

    elif mapping == "BK":
        if n_qubits is None:
            raise ValueError("n_qubits is required for BK.")

        try:
            qop = bravyi_kitaev(op, n_qubits=n_qubits)
        except TypeError:
            qop = bravyi_kitaev(op, n_qubits)

    else:
        raise ValueError("mapping must be 'JW' or 'BK'.")

    qop.compress(abs_tol=1e-12)
    return qop


def pauli_duplication_ratio(
    fermionic_terms,
    mapping="JW",
    n_qubits=None,
    include_identity=True,
    tol=1e-12,
):
    """
    Compute

        sum_alpha #mapping(H_alpha) / #mapping(H)

    where H = sum_alpha H_alpha.

    Also returns a duplicate-use table.
    """

    H_full = FermionOperator.zero()

    numerator = 0
    union_support = set()
    pauli_usage = defaultdict(list)

    for alpha, H_alpha in enumerate(fermionic_terms):
        H_full += H_alpha

        Q_alpha = map_fermion_to_qubit(
            H_alpha,
            mapping=mapping,
            n_qubits=n_qubits,
        )

        support_alpha = pauli_support(
            Q_alpha,
            tol=tol,
            include_identity=include_identity,
        )

        numerator += len(support_alpha)
        union_support |= support_alpha

        for pauli_key, coeff in Q_alpha.terms.items():
            if abs(coeff) <= tol:
                continue

            if not include_identity and pauli_key == ():
                continue

            pauli_usage[pauli_key].append(
                {
                    "vertex": f"T_{alpha}",
                    "coefficient": coeff,
                }
            )

    H_full = normal_ordered(H_full)
    H_full.compress(abs_tol=tol)

    Q_full = map_fermion_to_qubit(
        H_full,
        mapping=mapping,
        n_qubits=n_qubits,
    )

    full_support = pauli_support(
        Q_full,
        tol=tol,
        include_identity=include_identity,
    )

    denominator = len(full_support)

    duplication_ratio = numerator / denominator
    union_ratio = numerator / len(union_support)

    summary_df = pd.DataFrame(
        [
            {
                "mapping": mapping.upper(),
                "include_identity": include_identity,
                "sum_alpha_number_of_Pauli_strings": numerator,
                "number_of_unique_Pauli_strings_before_cancellation": len(union_support),
                "number_of_Pauli_strings_in_full_H": denominator,
                "duplication_ratio": duplication_ratio,
                "raw_reuse_ratio_before_cancellation": union_ratio,
            }
        ]
    )

    duplicate_rows = []

    for pauli_key, appearances in pauli_usage.items():
        if len(appearances) <= 1:
            continue

        combined_coeff = Q_full.terms.get(pauli_key, 0.0)

        duplicate_rows.append(
            {
                "Pauli_string": format_qubit_term(pauli_key),
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "individual_coefficients": [complex(x["coefficient"]) for x in appearances],
                "combined_coefficient_in_full_H": complex(combined_coeff),
                "survives_in_full_H": abs(combined_coeff) > tol,
            }
        )

    duplicate_df = pd.DataFrame(duplicate_rows)

    if len(duplicate_df) > 0:
        duplicate_df = duplicate_df.sort_values(
            "number_of_appearances",
            ascending=False,
        ).reset_index(drop=True)

    return summary_df, duplicate_df


# ------------------------------------------------------------
# Run for JW
# ------------------------------------------------------------

n_qubits = molecule.n_qubits

jw_summary_df, jw_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="JW",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== JW Pauli-duplication ratio ===")
display(jw_summary_df)

print("\n=== Duplicated JW Pauli strings ===")
display(jw_duplicate_df)


# ------------------------------------------------------------
# Optional: run for BK too
# ------------------------------------------------------------

bk_summary_df, bk_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="BK",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== BK Pauli-duplication ratio ===")
display(bk_summary_df)

print("\n=== Duplicated BK Pauli strings ===")
display(bk_duplicate_df)

=== JW Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,JW,True,16185,7921,4689,3.451695,2.043303



=== Duplicated JW Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,137,"[T_0, T_1, T_3, T_5, T_9, T_13, T_17, T_21, T_25, T_29, T_30, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_86, T_134, T_161, T_203, T_236, T_272, T_311, T_335, T_380, T_398, T_449, T_461, T_496, T_498, T_546, T_569, T_611, T_640, T_676, T_714, T_738, T_779, T_797, T_844, T_856, T_894, T_896, T_906, T_955, T_971, T_1012, T_1034, T_1068, T_1103, T_1131, T_1171, T_1192, T_1237, T_1251, T_1295, T_1299, T_1340, T_1356, T_1390, T_1412, T_1440, T_1475, T_1496, T_1536, T_1550, T_1593, T_1597, T_1607, T_1643, T_1658, T_1687, T_1710, T_1734, T_1761, T_1779, T_1810, T_1822, T_1853, T_1857, T_1886, T_1901, T_1925, T_1948, T_1966, T_1993, T_2005, T_2035, T_2039, T_2046, T_2071, T_2084, T_2104, T_2120, T_2135, T_2154, T_2164, ...]","[(-22.408915115183795+0j), (-2.0025723751678735+0j), (-2.0025723751678735+0j), (-1.8076608544243982+0j), (-1.8076608544243982+0j), (-1.8076608544243995+0j), (-1.8076608544243995+0j), (-1.8076608544243968+0j), (-1.8076608544243968+0j), (-1.4620212328253892+0j), (-1.4620212328253892+0j), (-1.462021232825395+0j), (-1.462021232825395+0j), (-1.4620212328254003+0j), (-1.4620212328254003+0j), (-1.367450382110078+0j), (-1.367450382110078+0j), (0.12601084193294032+0j), (0.09441778014524217+0j), (0.12150482690042828+0j), (0.09441778014524223+0j), (0.1215048269004284+0j), (0.09441778014524212+0j), (0.1215048269004282+0j), (0.11151132357155509+0j), (0.12341729243332238+0j), (0.11151132357155553+0j), (0.12341729243332288+0j), (0.11151132357155592+0j), (0.1234172924333233+0j), (0.09931008213958181+0j), (0.11766597355870295+0j), (0.12150482690042828+0j), (0.09441778014524217+0j), (0.1215048269004284+0j), (0.09441778014524223+0j), (0.1215048269004282+0j), (0.09441778014524212+0j), (0.12341729243332238+0j), (0.11151132357155509+0j), (0.12341729243332288+0j), (0.11151132357155553+0j), (0.1234172924333233+0j), (0.11151132357155592+0j), (0.11766597355870295+0j), (0.09931008213958181+0j), (0.13450436113594946+0j), (0.10134886265737106+0j), (0.11170989829963887+0j), (0.1013488626573709+0j), (0.10926625002654566+0j), (0.1064870678331826+0j), (0.1188088700377998+0j), (0.10569378505039045+0j), (0.1095313045443218+0j), (0.10275103248778099+0j), (0.13172862963369106+0j), (0.10338461264647633+0j), (0.11511605099515307+0j), (0.11170989829963887+0j), (0.10134886265737106+0j), (0.10926625002654566+0j), (0.1013488626573709+0j), (0.1188088700377998+0j), (0.1064870678331826+0j), (0.1095313045443218+0j), (0.10569378505039045+0j), (0.13172862963369106+0j), (0.10275103248778099+0j), (0.11511605099515307+0j), (0.10338461264647633+0j), (0.13581597072077464+0j), (0.10134886265737097+0j), (0.10795464044172066+0j), (0.10445999419292423+0j), (0.11562857121526804+0j), (0.10434289120617345+0j), (0.12895196049698127+0j), (0.10612899997225601+0j), (0.11548827250356242+0j), (0.10338461264647705+0j), (0.11511605099515433+0j), (0.10795464044172066+0j), (0.10134886265737097+0j), (0.11562857121526804+0j), (0.10445999419292423+0j), (0.12895196049698127+0j), (0.10434289120617345+0j), (0.11548827250356242+0j), (0.10612899997225601+0j), (0.11511605099515433+0j), (0.10338461264647705+0j), (0.1382596189938675+0j), (0.10398482334524581+0j), (0.12563136296274302+0j), (0.10489520911479004+0j), (0.12158553917450926+0j), (0.10605185291131804+0j), (0.11285190207856013+0j), (0.10338461264647632+0j), ...]",-35.283310+ 0.000000j,True
1,Z3,16,"[T_9, T_86, T_546, T_906, T_1299, T_1340, T_1356, T_1390, T_1412, T_1440, T_1475, T_1496, T_1536, T_1550, T_1593, T_1597]","[(1.8076608544243982+0j), (-0.12150482690042828+0j), (-0.09441778014524217+0j), (-0.13450436113594946+0j), (-0.11170989829963887+0j), (-0.10134886265737106+0j), (-0.10926625002654566+0j), (-0.1013488626573709+0j), (-0.1188088700377998+0j), (-0.1064870678331826+0j), (-0.1095313045443218+0j), (-0.10569378505039045+0j), (-0.13172862963369106+0j), (-0.10275103248778099+0j), (-0.11

=== BK Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,BK,True,16185,7921,4689,3.451695,2.043303



=== Duplicated BK Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,137,"[T_0, T_1, T_3, T_5, T_9, T_13, T_17, T_21, T_25, T_29, T_30, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_86, T_134, T_161, T_203, T_236, T_272, T_311, T_335, T_380, T_398, T_449, T_461, T_496, T_498, T_546, T_569, T_611, T_640, T_676, T_714, T_738, T_779, T_797, T_844, T_856, T_894, T_896, T_906, T_955, T_971, T_1012, T_1034, T_1068, T_1103, T_1131, T_1171, T_1192, T_1237, T_1251, T_1295, T_1299, T_1340, T_1356, T_1390, T_1412, T_1440, T_1475, T_1496, T_1536, T_1550, T_1593, T_1597, T_1607, T_1643, T_1658, T_1687, T_1710, T_1734, T_1761, T_1779, T_1810, T_1822, T_1853, T_1857, T_1886, T_1901, T_1925, T_1948, T_1966, T_1993, T_2005, T_2035, T_2039, T_2046, T_2071, T_2084, T_2104, T_2120, T_2135, T_2154, T_2164, ...]","[(-22.408915115183795+0j), (-2.0025723751678735+0j), (-2.0025723751678735+0j), (-1.8076608544243982+0j), (-1.8076608544243982+0j), (-1.8076608544243995+0j), (-1.8076608544243995+0j), (-1.8076608544243968+0j), (-1.8076608544243968+0j), (-1.4620212328253892+0j), (-1.4620212328253892+0j), (-1.462021232825395+0j), (-1.462021232825395+0j), (-1.4620212328254003+0j), (-1.4620212328254003+0j), (-1.367450382110078+0j), (-1.367450382110078+0j), (0.12601084193294032+0j), (0.09441778014524217+0j), (0.12150482690042828+0j), (0.09441778014524223+0j), (0.1215048269004284+0j), (0.09441778014524212+0j), (0.1215048269004282+0j), (0.11151132357155509+0j), (0.12341729243332238+0j), (0.11151132357155553+0j), (0.12341729243332288+0j), (0.11151132357155592+0j), (0.1234172924333233+0j), (0.09931008213958181+0j), (0.11766597355870295+0j), (0.12150482690042828+0j), (0.09441778014524217+0j), (0.1215048269004284+0j), (0.09441778014524223+0j), (0.1215048269004282+0j), (0.09441778014524212+0j), (0.12341729243332238+0j), (0.11151132357155509+0j), (0.12341729243332288+0j), (0.11151132357155553+0j), (0.1234172924333233+0j), (0.11151132357155592+0j), (0.11766597355870295+0j), (0.09931008213958181+0j), (0.13450436113594946+0j), (0.10134886265737106+0j), (0.11170989829963887+0j), (0.1013488626573709+0j), (0.10926625002654566+0j), (0.1064870678331826+0j), (0.1188088700377998+0j), (0.10569378505039045+0j), (0.1095313045443218+0j), (0.10275103248778099+0j), (0.13172862963369106+0j), (0.10338461264647633+0j), (0.11511605099515307+0j), (0.11170989829963887+0j), (0.10134886265737106+0j), (0.10926625002654566+0j), (0.1013488626573709+0j), (0.1188088700377998+0j), (0.1064870678331826+0j), (0.1095313045443218+0j), (0.10569378505039045+0j), (0.13172862963369106+0j), (0.10275103248778099+0j), (0.11511605099515307+0j), (0.10338461264647633+0j), (0.13581597072077464+0j), (0.10134886265737097+0j), (0.10795464044172066+0j), (0.10445999419292423+0j), (0.11562857121526804+0j), (0.10434289120617345+0j), (0.12895196049698127+0j), (0.10612899997225601+0j), (0.11548827250356242+0j), (0.10338461264647705+0j), (0.11511605099515433+0j), (0.10795464044172066+0j), (0.10134886265737097+0j), (0.11562857121526804+0j), (0.10445999419292423+0j), (0.12895196049698127+0j), (0.10434289120617345+0j), (0.11548827250356242+0j), (0.10612899997225601+0j), (0.11511605099515433+0j), (0.10338461264647705+0j), (0.1382596189938675+0j), (0.10398482334524581+0j), (0.12563136296274302+0j), (0.10489520911479004+0j), (0.12158553917450926+0j), (0.10605185291131804+0j), (0.11285190207856013+0j), (0.10338461264647632+0j), ...]",-35.283310+ 0.000000j,True
1,Z1 Z2 Z3,16,"[T_9, T_86, T_546, T_906, T_1299, T_1340, T_1356, T_1390, T_1412, T_1440, T_1475, T_1496, T_1536, T_1550, T_1593, T_1597]","[(1.8076608544243982+0j), (-0.12150482690042828+0j), (-0.09441778014524217+0j), (-0.13450436113594946+0j), (-0.11170989829963887+0j), (-0.10134886265737106+0j), (-0.10926625002654566+0j), (-0.1013488626573709+0j), (-0.1188088700377998+0j), (-0.1064870678331826+0j), (-0.1095313045443218+0j), (-0.10569378505039045+0j), (-0.13172862963369106+0j), (-0.10275103248778099+0j), 